# 03 · Tokens and tokenizers
Level L1 · Part 1 · core · ~25 min · builds on 02

## Goal
By the end of this lesson you can count the tokens in a Scout note with a real BPE tokenizer
(`tiktoken`, `o200k_base`), and explain why that count differs from the word count and from another model's count.

## The idea
In lesson 02 a token was a whole word. That breaks on the first word the model never saw (`Pluto`).
Real models use **subword tokens**: common words are one token, rare words are split into pieces,
and any text at all can be written with single bytes as a last resort.
A **BPE tokenizer** (byte-pair encoding) builds those pieces by starting from characters and
repeatedly gluing together the pair that appears most often. Each model family trains its own,
so the same note has a different token count on different models. You pay per token and the
context window is measured in tokens, so this count matters.

Analogy: text-message shorthand. Frequent phrases get a short code ("brb"); rare words are spelled out letter by letter.

## Picture
```mermaid
flowchart LR
    T["'Perseverance landed'"] --> S[Split into characters]
    S --> M["Apply learned merges in order<br/>for example e r -> er"]
    M --> P["Pieces: ' Perse' 'ver' 'ance' ' landed'"]
    P --> I["Token ids: numbers the model reads"]
```

## Step by step
### 1. Three Scout notes, three ways to count
We take Scout's six notes from lesson 02 and pick three with something tricky in them:
odd names (Phobos), numbers (24, 39), and a long rare word (Perseverance).
First the two counts anyone can do without a tokenizer: words and characters.

In [1]:
NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
PICKED = [NOTES[0], NOTES[2], NOTES[4]]          # notes 1, 3 and 5

for note in PICKED:
    t = note["text"]
    print(f"note {note['id']}: {len(t.split()):2} words, {len(t):2} characters")

note 1:  8 words, 44 characters
note 3: 11 words, 50 characters
note 5:  9 words, 55 characters


### 2. Why whole words fail
Lesson 02's tokenizer turned each word into a token. A model needs a fixed list of known
tokens, its **vocabulary**, with a number (id) for each. Here is a word vocabulary built from the six notes.

In [2]:
import re

def word_tokenize(text: str) -> list[str]:
    """Lesson 02's tokenizer: words and single punctuation marks."""
    return re.findall(r"[\w']+|[.,!?]", text)

word_vocab = sorted({tok for n in NOTES for tok in word_tokenize(n["text"])})
word_ids = {tok: i for i, tok in enumerate(word_vocab)}
print(len(word_vocab), "word tokens in the vocabulary")
print([word_ids[t] for t in word_tokenize("Mars has two small moons.")])

47 word tokens in the vocabulary
[9, 21, 45, 40, 31, 1]


Known words become ids. But `Pluto` has no id, and neither does any word outside these notes.
You could add every word in the world, but there is always one more (names, typos, code, emoji).
Characters are the opposite extreme: a tiny vocabulary that covers everything, but every note becomes ~50 tokens.
BPE sits in between.

### 3. Byte-pair encoding by hand: one merge
We split text into chunks first (each word keeps the space before it, like real tokenizers do),
then into single characters. Then we count every pair of neighbouring pieces.

In [3]:
from collections import Counter

def chunks(text: str) -> list[str]:
    """Words with their leading space, and punctuation."""
    return re.findall(r" ?\w+| ?[^\w\s]+", text)

TRAIN = [n["text"] for n in NOTES]
words = [tuple(c) for t in TRAIN for c in chunks(t)]
pairs = Counter(p for w in words for p in zip(w, w[1:]))
print(chunks(NOTES[0]["text"])[:4])
print(pairs.most_common(5))

['Mars', ' has', ' two', ' small']
[(('a', 'r'), 9), (('a', 'n'), 6), (('l', 'a'), 6), (('r', 's'), 5), ((' ', 'm'), 5)]


`a` followed by `r` is the most common pair (Mars, February, are, largest, ...). BPE's training rule:
**glue the most common pair into one new token**, rewrite the text with it, and count again.

In [4]:
def merge(word: tuple, a: str, b: str) -> tuple:
    """Replace every neighbouring (a, b) in word with the single piece a+b."""
    out, i = [], 0
    while i < len(word):
        if i < len(word) - 1 and word[i] == a and word[i + 1] == b:
            out.append(a + b)
            i += 2
        else:
            out.append(word[i])
            i += 1
    return tuple(out)

print(merge(tuple(" Mars"), "a", "r"))

(' ', 'M', 'ar', 's')


### 4. Train a toy BPE tokenizer
Repeat that step `n` times and keep the list of merges in order. That list *is* the trained tokenizer.

In [5]:
def train_bpe(texts: list[str], n_merges: int) -> list[tuple[str, str]]:
    words = [tuple(c) for t in texts for c in chunks(t)]
    merges = []
    for _ in range(n_merges):
        pairs = Counter(p for w in words for p in zip(w, w[1:]))
        if not pairs:
            break
        (a, b), _count = pairs.most_common(1)[0]
        merges.append((a, b))
        words = [merge(w, a, b) for w in words]
    return merges

merges_40 = train_bpe(TRAIN, 40)
print(merges_40[:10])

[('a', 'r'), ('a', 'n'), (' ', 'm'), ('i', 'n'), ('s', 't'), (' ', 's'), ('e', 'r'), ('M', 'ar'), ('Mar', 's'), (' ', 't')]


Notice `('M', 'ar')` then `('Mar', 's')`: after two merges `Mars` is a single token, because it appears in three notes.

### 5. Encode with the toy tokenizer
To tokenize new text, split it into characters and replay the merges in the order they were learned.

In [6]:
def bpe_encode(text: str, merges: list[tuple[str, str]]) -> list[str]:
    pieces = []
    for c in chunks(text):
        w = tuple(c)
        for a, b in merges:
            w = merge(w, a, b)
        pieces.extend(w)
    return pieces

EXTRA = ["Pluto is small.", "Le rover Perseverance a atterri sur Mars en février 2021.",
         "火星には二つの小さな衛星があります。", "3.14159265358979"]   # more test texts, used below
print(bpe_encode(PICKED[1]["text"], merges_40))
print(bpe_encode(EXTRA[0], merges_40))

['A', ' ', 'd', 'a', 'y', ' on', ' Mars', ' l', 'a', 'st', 's', ' ', 'a', 'bo', 'ut', ' 2', '4', ' h', 'o', 'ur', 's', ' and', ' ', '3', '9', ' m', 'in', 'ut', 'e', 's', '.']
['P', 'l', 'ut', 'o', ' i', 's', ' s', 'm', 'a', 'l', 'l', '.']


`Pluto` was never seen, yet it encodes fine: it falls back to smaller pieces (`P l ut o`). No word is ever unknown.
More merges means a bigger vocabulary and fewer tokens per note:

In [7]:
for n in [0, 10, 40]:
    merges = train_bpe(TRAIN, n)
    counts = [len(bpe_encode(note["text"], merges)) for note in PICKED]
    print(f"{n:3} merges -> tokens for notes 1, 3, 5: {counts}")

  0 merges -> tokens for notes 1, 3, 5: [44, 50, 55]
 10 merges -> tokens for notes 1, 3, 5: [37, 43, 45]
 40 merges -> tokens for notes 1, 3, 5: [28, 31, 32]


With 0 merges, tokens = characters. Real tokenizers do the same thing on bytes instead of
characters, trained on a huge amount of text, with 100,000 to 200,000 merges.

### 6. A real BPE tokenizer: tiktoken `o200k_base`
`tiktoken` is OpenAI's tokenizer library. `o200k_base` is the encoding used by recent OpenAI models.
The first call downloads its merge table (about 3.6 MB); we cache it in `../data/tiktoken_cache`.
This cell also saves its results to `../data/recorded/03.json` so the next cell works offline.

In [8]:
import json, os, pathlib
os.environ["TIKTOKEN_CACHE_DIR"] = "../data/tiktoken_cache"
import tiktoken

ENCODINGS = {name: tiktoken.get_encoding(name) for name in ["o200k_base", "cl100k_base"]}

def pieces_of(enc, text: str) -> list[str]:
    return [enc.decode_single_token_bytes(i).decode("utf-8", "replace") for i in enc.encode(text)]

recorded = {name: {t: pieces_of(enc, t) for t in [n["text"] for n in PICKED] + EXTRA}
            for name, enc in ENCODINGS.items()}
pathlib.Path("../data/recorded").mkdir(parents=True, exist_ok=True)
pathlib.Path("../data/recorded/03.json").write_text(json.dumps(recorded, ensure_ascii=False, indent=1))
print("o200k_base vocabulary size:", ENCODINGS["o200k_base"].n_vocab)
print("token ids for note 3:", ENCODINGS["o200k_base"].encode(PICKED[1]["text"]))

⏭  Skipped: this cell needs [download]. Run it yourself, or re-run with TUT_ENABLE=download.


Ids are just positions in the tokenizer's vocabulary; the model only ever sees these numbers.
The next cell gives us `real_tokens(encoding, text)`. If the download cell above was skipped, it loads
the **recorded output** from `../data/recorded/03.json` (real tiktoken results saved when this lesson was built).

In [9]:
import json, pathlib

if "recorded" not in globals():
    recorded = json.loads(pathlib.Path("../data/recorded/03.json").read_text())
    print("using recorded tiktoken output")

def real_tokens(encoding: str, text: str) -> list[str]:
    return recorded[encoding][text]

for note in PICKED:
    print(f"note {note['id']}:", real_tokens("o200k_base", note["text"]))

using recorded tiktoken output
note 1: ['Mars', ' has', ' two', ' small', ' moons', ',', ' Ph', 'obos', ' and', ' De', 'imos', '.']
note 3: ['A', ' day', ' on', ' Mars', ' lasts', ' about', ' ', '24', ' hours', ' and', ' ', '39', ' minutes', '.']
note 5: ['The', ' rover', ' Perse', 'ver', 'ance', ' landed', ' on', ' Mars', ' in', ' February', ' ', '202', '1', '.']


Read the pieces. `Mars`, `rover`, `landed` are one token each, with the space glued to the front.
`Phobos` is ` Ph` + `obos`, `Perseverance` is ` Perse` + `ver` + `ance`, and `2021` is `202` + `1`.
Frequent words get one token; rare words and numbers get split.

### 7. Words, characters, toy tokens, real tokens side by side

In [10]:
print(f"{'note':>4} {'words':>6} {'chars':>6} {'lesson02':>9} {'toyBPE40':>9} {'o200k':>6}")
for note in PICKED:
    t = note["text"]
    print(f"{note['id']:>4} {len(t.split()):>6} {len(t):>6} {len(word_tokenize(t)):>9}"
          f" {len(bpe_encode(t, merges_40)):>9} {len(real_tokens('o200k_base', t)):>6}")

note  words  chars  lesson02  toyBPE40  o200k
   1      8     44        10        28     12
   3     11     50        12        31     14
   5      9     55        10        32     14


For plain English, real tokens are a bit more than words (here 1.3 to 1.6 tokens per word) and
about a quarter of the characters. Our toy BPE is far worse: 40 merges learned from six sentences
cannot compete with 200,000 learned from a large share of the web.

### 8. Different tokenizers, different counts
`cl100k_base` is the older OpenAI encoding (about 100,000 tokens). Same text, two tokenizers:

In [11]:
for text in [PICKED[2]["text"]] + EXTRA[1:]:
    a = len(real_tokens("o200k_base", text))
    b = len(real_tokens("cl100k_base", text))
    print(f"words {len(text.split()):2} | o200k {a:2} | cl100k {b:2} | {text[:40]}")
print(real_tokens("o200k_base", EXTRA[2]))

words  9 | o200k 14 | cl100k 15 | The rover Perseverance landed on Mars in
words 10 | o200k 17 | cl100k 19 | Le rover Perseverance a atterri sur Mars
words  1 | o200k 13 | cl100k 17 | 火星には二つの小さな衛星があります。
words  1 | o200k  7 | cl100k  7 | 3.14159265358979
['火', '星', 'には', '二', 'つ', 'の', '小', 'さ', 'な', '衛', '星', 'があります', '。']


The English note barely changes (14 vs 15). French and Japanese gain more from the bigger vocabulary,
and the Japanese sentence is "1 word" to `split()` but 13 tokens. Word counts mean nothing outside English.
Long numbers are cut into groups of up to three digits (`141`, `592`, ...), so numbers cost more than they look.
Other vendors (Anthropic, Google) use their own tokenizers and do not publish them as tiktoken files,
so count their tokens with their own APIs (lesson 12). Even one vendor changes tokenizers between model versions.

## At work
- **Bills and limits are in tokens, never words.** Prices are quoted per million input and output tokens,
  and teams log the token `usage` the API returns on every call to track cost per turn. → lesson 12
- **A model upgrade can change your bill without any code change.** Anthropic's Claude 4.7+ models use a new
  tokenizer that produces about 30% more tokens for the same text, so teams re-count prompts per model before switching. → lesson 17
- **tiktoken is only correct for OpenAI models.** For Claude or Gemini, use the provider's token-counting
  endpoint (Anthropic's `messages.count_tokens` is free but rate-limited) or the `usage` field of the response. → lesson 12
- **The context window is a token budget.** Long notes, tool definitions and history all spend it, which is why
  you must count before you send. → lesson 05

## What just happened
- A word vocabulary had no id for `Pluto`; subword tokens fix that by falling back to smaller pieces.
- `train_bpe` learned merges by gluing the most common neighbouring pair, again and again; `bpe_encode` replays them.
- More merges gave fewer tokens per note (50 characters -> 31 tokens for note 3 with 40 merges).
- `tiktoken`'s `o200k_base` turned note 3 into 14 tokens: common words whole, `24` and `39` as their own tokens.
- `o200k_base` and `cl100k_base` gave different counts for the same text, most of all for French and Japanese.

## Common mistakes
**1. Estimating tokens from words.** It looks close for short English, then fails badly elsewhere.
Here a team checks the token count of a short Japanese note with `split()`:

In [12]:
note_jp = EXTRA[2]                                   # the Japanese sentence
estimate = len(note_jp.split())
actual = len(real_tokens("o200k_base", note_jp))
assert estimate == actual, f"split() says {estimate} token(s), the tokenizer says {actual}"

AssertionError: split() says 1 token(s), the tokenizer says 13

Fix: count with the tokenizer the model actually uses, `len(enc.encode(text))`, never `split()` or `len(text) / 4`.

**2. Using tiktoken for a non-OpenAI model.** tiktoken has no tokenizer for Claude, and says so:

In [13]:
import tiktoken
tiktoken.encoding_for_model("claude-opus-5-5")

KeyError: 'Could not automatically map claude-opus-5-5 to a tokeniser. Please use `tiktoken.get_encoding` to explicitly get the tokeniser you expect.'

Fix: for OpenAI models, `encoding_for_model("gpt-4o")` picks the right encoding. For other vendors, ask the
provider's API for the count (lesson 12). Picking `o200k_base` "as an estimate" for Claude would undercount.

## Try it
1. Train `train_bpe(TRAIN, 60)` and encode `"Phobos orbits Mars."`. How many tokens? Which pieces did it learn from the notes?
2. For all six `NOTES`, compute the toy BPE (40 merges) tokens-per-word ratio. Which note has the highest ratio, and why?

In [14]:
# your code here

## Key terms
- **token** — the unit of text a model reads and writes: a whole word, part of a word, a punctuation mark, or a byte.
- **vocabulary** — the fixed list of tokens a tokenizer knows, each with a number (token id).
- **token id** — the number for a token in the vocabulary; models read and write ids, not text.
- **subword token** — a token that is part of a word; rare words are split into several of them.
- **BPE tokenizer** — byte-pair encoding: starts from characters (or bytes) and repeatedly merges the most frequent neighbouring pair into a new token.
- **merge** — one learned BPE rule "glue piece a and piece b into ab"; a tokenizer is an ordered list of merges.
- **encoding (tiktoken)** — a named, trained tokenizer such as `o200k_base` or `cl100k_base`.
- **tokenizer differences** — each model family (and sometimes each version) has its own tokenizer, so the same text has different token counts.